In [ ]:
import math
import time
import torch

# Minimal setup
vocab_size = 100
d_model = 64
d_k = 64
d_v = 64

torch.manual_seed(42)

emb = torch.randn(vocab_size, d_model)
w_q = torch.randn(d_model, d_k)
w_k = torch.randn(d_model, d_k)
w_v = torch.randn(d_model, d_v)
w_u = torch.randn(vocab_size, d_v)


def attention(q, k, v):
    s = q @ k.transpose(-1, -2) / math.sqrt(q.size(-1))
    ahead = torch.arange(len(k)) > torch.arange(len(k) - len(q), len(k))[:, None]
    s = s.masked_fill(ahead, float('-inf'))
    a = torch.softmax(s, dim=-1)
    return a @ v


def prefill(ids):
    x = emb[ids]
    return x @ w_k, x @ w_v


# WITH KV CACHE
def generate(ids, n_new, k_cache, v_cache):
    # Prefill stage
    k, v = prefill(ids)
    L = len(ids)
    k_cache[:L] = k
    v_cache[:L] = v

    for _ in range(n_new):
        q_new = emb[ids[-1:]] @ w_q
        # Only use the valid part of the cache
        o = attention(q_new, k_cache[:L], v_cache[:L])
        logits = o @ w_u.T
        next_id = int(logits.argmax())
        ids = ids + [next_id]

        # Compute K, V ONLY for the new token
        k_new, v_new = prefill([next_id])
        # Directly write into the pre-allocated cache
        k_cache[L] = k_new
        v_cache[L] = v_new
        L += 1

    return ids


# WITHOUT KV CACHE
def generate_nocache(ids, n_new, k_cache, v_cache):
    for _ in range(n_new):
        q_new = emb[ids[-1:]] @ w_q
        # Recompute K, V for the ENTIRE sequence every step
        k, v = prefill(ids)
        L = len(ids)
        k_cache[:L] = k
        v_cache[:L] = v
        o = attention(q_new, k_cache[:L], v_cache[:L])
        logits = o @ w_u.T
        next_id = int(logits.argmax())
        ids = ids + [next_id]

    return ids


if __name__ == "__main__":
    prompt = [5, 12, 28]
    n_new = 2000
    max_len = len(prompt) + n_new + 10

    # Pre-allocate identical caches for both functions
    k_cache = torch.zeros(max_len, d_k)
    v_cache = torch.zeros(max_len, d_v)

    print(f"Initial Prompt: {prompt}")
    print("-" * 40)

    # Warmup
    generate(prompt.copy(), 10, k_cache.clone(), v_cache.clone())
    generate_nocache(prompt.copy(), 10, k_cache.clone(), v_cache.clone())

    # 1. WITH CACHE
    k_cache_1 = torch.zeros(max_len, d_k)
    v_cache_1 = torch.zeros(max_len, d_v)
    start = time.perf_counter()
    result_cache = generate(prompt.copy(), n_new, k_cache_1, v_cache_1)
    time_cache = time.perf_counter() - start

    # 2. WITHOUT CACHE
    k_cache_2 = torch.zeros(max_len, d_k)
    v_cache_2 = torch.zeros(max_len, d_v)
    start = time.perf_counter()
    result_nocache = generate_nocache(prompt.copy(), n_new, k_cache_2, v_cache_2)
    time_nocache = time.perf_counter() - start

    print(f"Generate WITH KV Cache time:    {time_cache:.4f} seconds")
    print(f"Generate WITHOUT KV Cache time: {time_nocache:.4f} seconds")

    if result_cache == result_nocache:
        print("\nVerification Passed: Both methods produce identical results!")
    else:
        print("\nVerification Failed: Results are different!")

    print(f"\nSpeedup: {time_nocache / time_cache:.2f}x faster with KV Cache.")